# Python Interview Cheat Sheet

> A compact, practical revision guide for Python interviews. Run examples in Python 3.10+ unless noted.

## 1. Core Syntax

```python
# Variables are dynamically typed
name = "Suraj"; age = 31

# Indentation defines code blocks (usually 4 spaces)
if age >= 18:
    print("Adult")

# Comments and multi-line strings
# one-line comment
"""Often used as a docstring."""
```

**Naming:** `snake_case` for functions/variables, `PascalCase` for classes, `UPPER_CASE` for constants.

## 2. Data Types, Mutability, and Hashing

| Type | Example | Mutable? | Hashable? |
|---|---|---:|---:|
| `int`, `float`, `bool`, `complex` | `10`, `3.2`, `True` | No | Yes |
| `str`, `bytes` | `"hi"`, `b"hi"` | No | Yes |
| `tuple` | `(1, 2)` | No | Usually* |
| `frozenset` | `frozenset({1, 2})` | No | Yes |
| `list`, `dict`, `set`, `bytearray` | `[]`, `{}`, `set()` | Yes | No |

`*` A tuple is hashable only when **every element inside it is hashable**.

```python
hash((1, "x"))       # works
hash(([1, 2], 3))     # TypeError: list is unhashable
```

**Memory trick:** mutable objects can change, so their hash must not be used as a stable dictionary/set key. “Mutable → not hashable”; “immutable → usually hashable.”

### Important collection facts

```python
lst = [1, 2, 2]       # ordered, allows duplicates
tup = (1, 2)          # ordered, immutable
st = {1, 2, 2}        # unique values, unordered/no index
d = {"name": "Ada"}  # key → value mapping; keys must be hashable

empty_dict = {}
empty_set = set()     # {} is NOT an empty set
```

## 3. Operators

```python
# Arithmetic: + - * / // % **
7 / 2     # 3.5      true division
7 // 2    # 3        floor division
7 % 2     # 1        remainder
2 ** 3    # 8        power

# Comparison: == != < <= > >=
# Logical: and, or, not (short-circuit)
# Membership: in, not in
# Identity: is, is not
```

```python
a = [1, 2]; b = [1, 2]; c = a
a == b    # True: same value
a is b    # False: different objects
a is c    # True: same object
```

Use `is None`, not `== None`.

## 4. Control Flow

```python
if score >= 90:
    grade = "A"
elif score >= 60:
    grade = "B"
else:
    grade = "C"

for i, value in enumerate(["a", "b"], start=1):
    print(i, value)

while condition:
    break       # leave loop
    continue    # next iteration

for item in items:
    ...
else:           # runs only if loop did NOT break
    print("completed")
```

```python
match command:          # Python 3.10+
    case "start": print("Go")
    case _: print("Unknown")
```

## 5. Functions and Arguments

```python
def greet(name, greeting="Hello"):
    """Return a greeting."""
    return f"{greeting}, {name}!"

greet("Ada")
greet(name="Ada", greeting="Hi")
```

### Argument kinds

```python
def demo(pos_only, /, normal, default=0, *args, kw_only, **kwargs):
    return pos_only, normal, default, args, kw_only, kwargs

demo(1, 2, 3, 4, 5, kw_only=6, city="Pune")
```

| Form | Meaning |
|---|---|
| positional | matched by order |
| keyword | matched by parameter name |
| `/` | parameters before it are positional-only |
| `*args` | extra positional arguments, collected into a **tuple** |
| `*` | later parameters are keyword-only |
| `**kwargs` | extra named arguments, collected into a **dict** |

**Rule:** positional arguments come before keyword arguments in a call.

```python
nums = [2, 3]
options = {"greeting": "Hi"}
greet(*["Ada"], **options)  # unpack list/tuple with *, dict with **
```

### Avoid mutable default arguments

```python
# Bad: one list is shared across calls
def add_bad(x, bucket=[]):
    bucket.append(x); return bucket

# Good
def add(x, bucket=None):
    if bucket is None:
        bucket = []
    bucket.append(x)
    return bucket
```

### Lambda, `map`, `filter`, `reduce`

```python
square = lambda x: x * x
list(map(square, [1, 2, 3]))               # [1, 4, 9]
list(filter(lambda x: x % 2 == 0, [1, 2])) # [2]

from functools import reduce
reduce(lambda a, b: a + b, [1, 2, 3])      # 6
```

Prefer a named `def` when logic is non-trivial; comprehensions are often clearer than `map`/`filter`.

## 6. Scope, Closures, and LEGB

Python resolves names in this order: **L**ocal → **E**nclosing → **G**lobal → **B**uilt-in.

```python
x = "global"
def outer():
    x = "enclosing"
    def inner():
        nonlocal x       # modify enclosing x
        x = "changed"
    inner()
    return x
```

Use `global x` only when you truly need to rebind a module-level name.

```python
def make_multiplier(n):
    def multiply(x):     # closure: remembers n
        return x * n
    return multiply

double = make_multiplier(2)
double(5)  # 10
```

## 7. Comprehensions

```python
squares = [x * x for x in range(6)]
evens = [x for x in range(10) if x % 2 == 0]
lengths = {word: len(word) for word in ["cat", "python"]}
unique_lengths = {len(word) for word in ["hi", "cat", "go"]}
generator = (x * x for x in range(1_000_000))  # lazy
```

Nested comprehension:

```python
flat = [item for row in [[1, 2], [3, 4]] for item in row]
```

Read it left to right: “for each row, for each item in that row.” Keep deeply nested logic as normal loops for readability.

## 8. Iterables, Iterators, and Generators

| Term | Meaning |
|---|---|
| Iterable | Can produce an iterator: list, string, dict, file |
| Iterator | Has `__next__()` and returns items one at a time |
| Generator | Easy iterator created with `yield` or a generator expression |

```python
it = iter([10, 20])
next(it)  # 10
next(it)  # 20
# next(it) raises StopIteration

def countdown(n):
    while n:
        yield n          # pauses here, preserving state
        n -= 1

for n in countdown(3):
    print(n)
```

**Memory trick:** a list makes all values now; a generator makes the next value only when asked.

## 9. Decorators

A decorator adds behavior around a function without editing that function’s source.

```python
from functools import wraps

def logger(func):
    @wraps(func)         # preserves name/docstring/metadata
    def wrapper(*args, **kwargs):
        print(f"Calling {func.__name__}")
        result = func(*args, **kwargs)
        print("Done")
        return result
    return wrapper

@logger
def add(a, b):
    return a + b

# @logger means: add = logger(add)
```

Common uses: logging, timing, authorization, retries, caching (`@functools.cache`).

## 10. Exception Handling

```python
try:
    result = 10 / divisor
except ZeroDivisionError:
    print("Cannot divide by zero")
except (TypeError, ValueError) as err:
    print(f"Invalid input: {err}")
else:
    print("Success:", result)    # only when try succeeds
finally:
    print("Cleanup always runs")  # even with return/exception
```

```python
raise ValueError("age must be positive")

class InvalidAgeError(ValueError):
    pass
```

Avoid broad `except Exception` unless you can handle every error meaningfully; never use bare `except:` in ordinary application code. A `return` inside `try` skips `else`, but `finally` still runs.

## 11. File Handling and Context Managers

```python
from pathlib import Path

path = Path("notes.txt")
with path.open("w", encoding="utf-8") as file:
    file.write("Hello\n")

with path.open("r", encoding="utf-8") as file:
    text = file.read()
```

| Mode | Meaning |
|---|---|
| `r` | read; file must exist |
| `w` | write; creates or truncates |
| `a` | append; creates if missing |
| `x` | create; fails if already exists |
| `b` | binary mode (`rb`, `wb`) |

`with` guarantees cleanup (such as closing a file), even if an exception happens.

## 12. Modules and Packages

```python
import math
from math import sqrt
import numpy as np

if __name__ == "__main__":
    main()  # runs only when this file is executed directly
```

- A **module** is a `.py` file.
- A **package** is a directory of modules (commonly with `__init__.py`; namespace packages are also possible).
- Prefer `import module` or clear aliases over `from module import *`.

## 13. OOP: Class, Object, and Methods

```python
class Student:
    college = "ABC"                    # class variable: shared

    def __init__(self, name, age):      # constructor/initializer
        self.name = name                # instance variables: per object
        self.age = age

    def introduce(self):                # instance method: receives self
        return f"I am {self.name}"

    @classmethod
    def change_college(cls, college):   # receives class
        cls.college = college

    @staticmethod
    def is_adult(age):                  # no self or cls needed
        return age >= 18
```

| Method | First argument | Best for |
|---|---|---|
| Instance | `self` | object-specific data/behavior |
| Class | `cls` | class state, alternate constructors |
| Static | none | related utility with no object/class state |

### Four pillars

```python
# Inheritance + overriding + super()
class Animal:
    def speak(self): return "sound"

class Dog(Animal):
    def speak(self): return "bark"     # overriding

# Polymorphism / duck typing: same required behavior, different classes
def make_sound(animal):
    return animal.speak()
```

- **Inheritance:** child reuses/extends parent (`Dog is an Animal`).
- **Encapsulation:** keep implementation details controlled; `_x` is conventionally internal, `__x` triggers name mangling.
- **Abstraction:** expose a required interface, hide implementation.
- **Polymorphism:** one interface, multiple implementations.

```python
from abc import ABC, abstractmethod

class Payment(ABC):
    @abstractmethod
    def pay(self, amount):
        """Subclasses must implement this."""
```

Prefer **composition** for a “has-a” relationship (`Car has an Engine`) and inheritance for a real “is-a” relationship.

### Properties and validation

```python
class Temperature:
    def __init__(self, celsius): self.celsius = celsius

    @property
    def celsius(self): return self._celsius

    @celsius.setter
    def celsius(self, value):
        if value < -273.15: raise ValueError("below absolute zero")
        self._celsius = value
```

### Multiple inheritance and MRO

```python
class C(A, B): pass
C.mro()  # lookup order; Python uses C3 linearization
```

Use cooperative `super()` in every class that participates in a multiple-inheritance hierarchy.

## 14. Dunder (Magic) Methods

Special methods let your objects work with Python syntax and built-ins.

```python
class Vector:
    def __init__(self, x, y): self.x, self.y = x, y
    def __repr__(self): return f"Vector({self.x}, {self.y})"
    def __str__(self): return f"({self.x}, {self.y})"
    def __len__(self): return 2
    def __add__(self, other): return Vector(self.x + other.x, self.y + other.y)
    def __eq__(self, other): return isinstance(other, Vector) and (self.x, self.y) == (other.x, other.y)
```

| Method | Used by |
|---|---|
| `__init__` | object initialization |
| `__new__` | object creation (advanced; immutable subclasses) |
| `__str__` / `__repr__` | `str(obj)` / developer representation |
| `__len__`, `__iter__`, `__getitem__` | `len`, iteration, indexing |
| `__eq__`, `__lt__` | comparisons |
| `__add__` | `+` operator |
| `__enter__`, `__exit__` | `with` statement |

`__repr__` should ideally be unambiguous/re-creatable; `__str__` is friendly for users. If you define equality on a mutable custom class, understand the `__hash__` contract before using it as a dictionary key.

## 15. Useful `collections` Types

```python
from collections import Counter, defaultdict, deque, namedtuple

Counter("banana").most_common(1)              # [('a', 3)]
groups = defaultdict(list); groups["A"].append("Ada")
queue = deque([1, 2]); queue.appendleft(0); queue.pop()
Point = namedtuple("Point", "x y"); Point(1, 2).x
```

Also know `dataclasses`:

```python
from dataclasses import dataclass

@dataclass
class User:
    name: str
    active: bool = True
```

It generates common methods such as `__init__` and `__repr__`.

## 16. Type Hints

Hints improve readability and enable static checking; Python does not enforce them automatically at runtime.

```python
from collections.abc import Iterable

def mean(values: Iterable[float]) -> float:
    values = list(values)
    return sum(values) / len(values)

def find_user(user_id: int) -> str | None:
    return None
```

Common forms: `list[int]`, `dict[str, float]`, `tuple[int, str]`, `str | None`, `Callable[[int], str]`, `Any`, `TypeVar`, `Protocol`.

## 17. Common Built-ins to Know

```python
len(items); type(obj); isinstance(obj, str)
range(1, 10, 2); enumerate(items); zip(names, scores)
sorted(items, key=len, reverse=True); reversed(items)
sum(nums); min(nums); max(nums); any(flags); all(flags)
abs(-3); round(3.14159, 2); divmod(7, 2)
list("abc"); tuple(items); set(items); dict(pairs)
```

```python
people = [{"name": "Ada", "age": 36}, {"name": "Lin", "age": 30}]
sorted(people, key=lambda person: person["age"])
```

`zip()` stops at the shortest input. Use `zip(..., strict=True)` (Python 3.10+) when unequal lengths are an error.

## 18. Copying and Object References

```python
import copy
a = [[1], [2]]
b = a.copy()            # shallow: nested lists are still shared
c = copy.deepcopy(a)    # deep: recursively copied

b[0].append(99)
# a is now [[1, 99], [2]]
```

Assignment (`b = a`) creates another reference, not a copy.

## 19. NumPy vs Pandas

| Topic | NumPy | Pandas |
|---|---|---|
| Main structure | `ndarray` | `Series`, `DataFrame` |
| Best for | fast numerical/matrix work | labeled tabular data analysis |
| Values | generally homogeneous | columns may have different types |
| Labels/missing data | basic | rich indexing and missing-data tools |
| Typical use | vectorized math, linear algebra | CSV/Excel, cleaning, grouping, joins |

```python
import numpy as np
arr = np.array([1, 2, 3])
arr * 2                         # [2, 4, 6], elementwise

import pandas as pd
df = pd.DataFrame({"name": ["Ada", "Lin"], "score": [90, 80]})
df[df["score"] >= 85]
df.groupby("name")["score"].mean()
```

**Memory trick:** **NumPy = Numbers**, **Pandas = Tables**. Pandas commonly relies on NumPy (and may use other array backends in modern versions).

## 20. Common Interview Pitfalls

1. `==` compares values; `is` compares object identity. Use `is None`.
2. `list.sort()` changes a list and returns `None`; `sorted()` returns a new list.
3. `dict.get("x")` returns `None` by default; distinguish missing from a stored `None` when needed.
4. Do not mutate a list while iterating over it; iterate over `items[:]`, build a new list, or iterate backwards.
5. A `set` has no reliable index/order API; use a list when order/duplicates matter.
6. Dictionary insertion order is guaranteed in Python 3.7+; do not confuse that with hash sorting.
7. Strings are immutable: repeated `+=` in a large loop is inefficient; use `"".join(parts)`.
8. `and`/`or` return an operand, not always `True`/`False`: `"ok" and 5` is `5`.
9. Late binding in closures: lambdas in a loop see the final loop variable; bind it as a default argument.
10. `finally` can accidentally hide exceptions/returns if it itself returns—avoid returning from `finally`.

```python
# Late-binding fix
funcs = [lambda i=i: i for i in range(3)]
[f() for f in funcs]  # [0, 1, 2]
```

## 21. Rapid Revision: 20-Second Answers

| Question | Interview answer |
|---|---|
| List vs tuple? | List is mutable; tuple is immutable and can be hashable if all its elements are hashable. |
| `*args` vs `**kwargs`? | Extra positional arguments in a tuple vs extra keyword arguments in a dictionary. |
| Generator vs list? | Generator is lazy and memory-efficient; list stores all values immediately. |
| Decorator? | A callable that wraps another callable to add behavior without changing its source. |
| `try` / `else` / `finally`? | `else` runs only on success; `finally` runs regardless for cleanup. |
| `self` vs `cls`? | `self` is the instance; `cls` is the class. |
| Overloading vs overriding? | Python supports overriding; traditional signature-based overloading is usually simulated with defaults or `*args`. |
| Shallow vs deep copy? | Shallow copies outer container only; deep recursively copies nested objects. |
| NumPy vs Pandas? | NumPy for fast numerical arrays; Pandas for labeled tables and data analysis. |

## Final Memory Map

```text
Mutable: list, dict, set        → not hashable
*args: tuple                    **kwargs: dict
LEGB: Local → Enclosing → Global → Built-in
try: risky | except: error | else: success | finally: always
List: now / all values          Generator: lazy / next value
self: object                   cls: class
NumPy: numerical arrays         Pandas: labeled tables
```
